# FlowDataLab · AWARE Prioritisation Challenge — notebook destilado

**Qué es:** el resumen de todo el proyecto en un solo sitio: qué dicen los datos, de dónde sale la nota, por qué unos modelos
funcionan y otros no, dónde está el techo y por qué elegimos las entregas que elegimos. El detalle de cada prueba vive en
`futuro/LGB_futuro.ipynb` (diario de la línea con futuro), `notebooks/01_eda.ipynb` (EDA), `notebooks/LGB_adri.ipynb` y
`notebooks/NN_pablo.ipynb` (línea limpia), `realista/` (línea sin futuro) y `futuro/pruebas/resultados.jsonl` (registro de ejecuciones).

**Cómo se ejecuta:** de arriba abajo, en unos 2–3 minutos. No entrena nada: calcula lo barato desde los datos (`Data/`), los OOF
(`oof/`, `futuro/oof`, `futuro/pruebas/oof`, `realista/oof`) y los envíos; lo caro lo lee de los registros. Si falta un archivo,
usa la cifra registrada y lo indica con `(registro)`.

**Vocabulario.** *Snapshot* = una muestra (`sample_id`): la lista de tareas abiertas del controlador en un instante (9–59, ~36).
*Racha* = snapshots seguidos de la misma sesión sin huecos de más de 4.5 s. *Tramo* = racha × tarea correcta.
*CV* = top-3 hit rate fuera de fold con `Data/folds.csv` (5 folds, agrupados por controlador: cada controlador se predice con un
modelo que no lo ha visto). *Público* = controlador 020 (2 sesiones); *privado*, el que decide = controlador 023 (3 sesiones).

In [1]:
import sys, os, json, numpy as np, pandas as pd
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "Data").exists():
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT / "cv"), str(ROOT / "futuro"), str(ROOT / "futuro" / "pruebas")]
from metric import hits
from base_fut import datos
pd.set_option("display.width", 200, "display.max_colwidth", 80)

lab, te, fs = datos()          # train+validation y test con las features de futuro (cache en futuro/pruebas/cache.parquet)
TY = ['entry_conflict_resolution', 'exit_conflict_resolution', 'entry_coordination', 'exit_coordination',
      'assume_aircraft', 'transfer_aircraft', 'conflict_resolution', 'non_conformance_resolution', 'qos_improvement',
      'entry_conditions', 'exit_conditions', 'aircraft_request', 'zone_conflict', 'return_to_route']
snap = lab.drop_duplicates("sample_id").set_index("sample_id")[["session", "scenario", "atco", "fold", "seq_len", "block"]]
resp = lab[lab.target_index == 1].set_index("sample_id")

def oof(nombre, carpeta="oof"):
    for d in ("oof", "futuro/oof", "futuro/pruebas/oof", "realista/oof") if carpeta == "oof" else \
             ("preds", "futuro/preds", "futuro/pruebas/preds", "realista/preds"):
        p = ROOT / d / f"{nombre}.csv"
        if p.exists():
            return pd.read_csv(p)
    return None

def registro(nombre):
    for f in (ROOT / "futuro/pruebas/resultados.jsonl", ROOT / "pruebas/resultados.jsonl"):
        for l in open(f):
            x = json.loads(l)
            if x["nombre"] == nombre:
                r = x
    return locals().get("r")

def medir(nombre):
    o = oof(nombre)
    if o is None:
        r = registro(nombre)
        return dict(cv=r and r["cv"], peor_controlador=r and r["min_atco"], mediana_sesion=None, fuente="(registro)")
    h = hits(o).rename("hit").to_frame().join(snap)
    return dict(cv=round(h.hit.mean(), 4), peor_controlador=round(h.groupby("atco").hit.mean().min(), 3),
                mediana_sesion=round(h.groupby("session").hit.median().median() if False else h.groupby("session").hit.mean().median(), 4),
                fuente="OOF")
print(f"{lab.sample_id.nunique()} snapshots etiquetados, {te.sample_id.nunique()} de test")

13175 snapshots etiquetados, 1863 de test


## 1 · Los datos

Cada fila es una tarea del snapshot; exactamente una tiene `target_index = 1`. Train + validation: 18 controladores,
37 sesiones, 3 escenarios de tráfico (todas las sesiones de un escenario tienen los mismos aviones). Test: 020 y 023.

In [2]:
est = pd.DataFrame({
    "snapshots": [lab.sample_id.nunique(), te.sample_id.nunique()],
    "filas": [len(lab), len(te)],
    "sesiones": [lab.session.nunique(), te.session.nunique()],
    "controladores": [lab.atco.nunique(), te.atco.nunique()],
    "tareas por snapshot (media)": [round(lab.groupby("sample_id").size().mean(), 1), round(te.groupby("sample_id").size().mean(), 1)],
}, index=["etiquetado", "test"])
est

,snapshots,filas,sesiones,controladores,tareas por snapshot (media)
etiquetado,13175,474688,37,18,36.0
test,1863,66409,5,2,35.6


### 1a · Qué tipos de tarea se eligen: el *lift*

Lift = % de las respuestas / % de las filas. Transfer y assume son pocas filas y muchas respuestas; las coordinaciones
(entry/exit) son muchas filas y pocas respuestas: casi siempre hay alguna en la lista y casi nunca es la que se trabaja.

In [3]:
filas = lab.task_index.value_counts(normalize=True)
resps = resp.task_index.value_counts(normalize=True)
lift = pd.DataFrame({"% filas": filas * 100, "% respuestas": resps * 100}).fillna(0)
lift["lift"] = lift["% respuestas"] / lift["% filas"]
lift.index = [TY[i] for i in lift.index]
lift.sort_values("% respuestas", ascending=False).round(2)

,% filas,% respuestas,lift
transfer_aircraft,6.37,26.77,4.20
assume_aircraft,16.62,25.55,1.54
zone_conflict,11.19,16.10,1.44
entry_coordination,29.82,8.11,0.27
return_to_route,3.48,7.11,2.04
exit_coordination,12.68,5.65,0.45
exit_conditions,7.72,3.40,0.44
conflict_resolution,2.33,2.96,1.27
qos_improvement,6.30,1.95,0.31
exit_conflict_resolution,1.28,1.50,1.17


### 1b · La estructura de las etiquetas

Dos regularidades que no son de un snapshot sino de la sesión, y que explican buena parte del modelo final:
1. **Dentro de una racha el controlador sigue con la misma tarea.** Tras un hueco, casi nunca vuelve a ella.
2. **Cada tarea se trabaja en un solo episodio de la sesión.**

In [4]:
s = snap.join(resp[["task_name", "t"]]).sort_values(["session", "t"])
s["prev_task"] = s.groupby("session").task_name.shift()
s["prev_block"] = s.groupby("session").block.shift()
dentro = s[s.block == s.prev_block]
tras_hueco = s[(s.block != s.prev_block) & s.prev_block.notna()]
ep = s.groupby(["session", "task_name"]).block.nunique()
pd.Series({
    "repite la respuesta del snapshot anterior, dentro de la racha": f"{(dentro.task_name == dentro.prev_task).mean():.1%}",
    "repite la respuesta tras un hueco entre rachas": f"{(tras_hueco.task_name == tras_hueco.prev_task).mean():.1%}",
    "tareas que son respuesta en una sola racha de la sesión": f"{(ep == 1).mean():.1%}",
    "snapshots por racha (media)": f"{snap.groupby('block').size().mean():.1f}",
}).to_frame("valor")

,valor
"repite la respuesta del snapshot anterior, dentro de la racha",94.8%
repite la respuesta tras un hueco entre rachas,0.2%
tareas que son respuesta en una sola racha de la sesión,98.4%
snapshots por racha (media),4.2


### 1c · Lo que distingue a la tarea elegida (dentro del propio snapshot)

Del EDA (`notebooks/01_eda.ipynb`, sección 5–6), estable en los 5 folds:
- La elegida está **más cerca del borde del sector** (`time_to_sector`, `k_timeToExitPoint`, rango `l_rank_absf1_*`): assume justo
  antes de entrar, transfer justo antes de salir. Es la más cercana dentro de su tipo el 70–76 % de las veces (transfer).
- Tiene **menos vacíos**: los `k_*` están vacíos cuando el avión no está asumido (los vacíos significan "no aplica").
- Un ranker de una sola columna (prior por tipo + rango de cercanía al borde) ya da **0.41**; el azar da 0.09.
- **Twin** assume + entry_coordination del mismo avión: el 29.9 % de las respuestas; se elige assume el 68–78 %.
- El **escenario** pesa más que el controlador en el tipo elegido (Cramér's V 0.159 frente a 0.096).

In [5]:
# comprobación rápida: la elegida es la más cercana al borde dentro de su tipo
x = lab[lab.task_index.isin([4, 5])].copy()
x["rk"] = x.assign(a=x.time_to_sector.abs()).groupby(["sample_id", "task_index"]).a.rank(method="first")
r = x[x.target_index == 1].groupby("task_index").rk.apply(lambda v: (v == 1).mean())
r.index = [TY[i] for i in r.index]
r.rename("P(la elegida es la más cercana al borde de su tipo)").round(3).to_frame()

,P(la elegida es la más cercana al borde de su tipo)
assume_aircraft,0.684
transfer_aircraft,0.724


## 2 · De dónde sale la nota: tres líneas y cuánta información aporta cada fuente

| Línea | Qué información usa | Dónde |
|---|---|---|
| **limpia** (`main`) | solo las filas del propio snapshot | `src/`, `notebooks/LGB_adri.ipynb`, `oof/` |
| **realista** | snapshot + snapshots **pasados** de la sesión | `realista/` |
| **futuro** | snapshot + pasado + **futuro** de la sesión (+ opcionalmente lo que hicieron otros controladores) | `futuro/` |

La tabla siguiente es la cadena completa, medida con los mismos folds.

In [6]:
CADENA = [
    ("limpia", "lgb_adri", "LightGBM LambdaRank, columnas originales + alerta + contexto"),
    ("limpia", "stack_meta", "meta-ranker sobre lgb_adri + red + tipos"),
    ("limpia", "lgb_adri_v2", "+ peso por tramo, 10 semillas (Adri) · público 0.642"),
    ("realista", "real_adri_noche_ses", "+ pasado: causal (cambios de estado recientes) + geo, búsqueda de parámetros (Pablo, Adri)"),
    ("futuro", "lgb_future", "+ futuro de la sesión: siguiente instrucción al avión, desaparición de la tarea · público 0.942"),
    ("futuro", "lgb_future_clear", "+ bloque clear: cuándo cambia el nivel, rumbo, zona o ruta del avión"),
    ("futuro", "lgb_future_clear_ses", "+ segundo paso de sesión: cada tarea se trabaja una sola vez"),
    ("futuro", "suav_lgb_future_clear10_ses", "A: 10 semillas + suavizado en la racha · público 0.947"),
    ("futuro", "f10_instr_l63_s10_ses", "N: + bloque instr (tipo de instrucción que encaja con la tarea) + 63 hojas + extra_trees · público 0.951"),
    ("futuro+oráculo", "suav_lgb_future_clear10_orcoord_ses", "B: A + oráculo de coordinaciones (otros controladores) · público 0.934"),
    ("futuro+oráculo", "f10_instr_l63_or_s10_ses", "NOR: N + oráculo de coordinaciones · público 0.948"),
]
T = pd.DataFrame([dict(linea=l, modelo=m, que=q, **medir(m)) for l, m, q in CADENA])
T

,linea,modelo,que,cv,peor_controlador,mediana_sesion,fuente
0,limpia,lgb_adri,"LightGBM LambdaRank, columnas originales + alerta + contexto",0.6562,0.571,0.6627,OOF
1,limpia,stack_meta,meta-ranker sobre lgb_adri + red + tipos,0.6600,0.580,0.6643,OOF
2,limpia,lgb_adri_v2,"+ peso por tramo, 10 semillas (Adri) · público 0.642",0.6644,0.579,0.6783,OOF
3,realista,real_adri_noche_ses,"+ pasado: causal (cambios de estado recientes) + geo, búsqueda de parámetros...",0.6722,0.595,0.6763,OOF
4,futuro,lgb_future,"+ futuro de la sesión: siguiente instrucción al avión, desaparición de la ta...",0.9010,0.769,0.9073,OOF
5,futuro,lgb_future_clear,"+ bloque clear: cuándo cambia el nivel, rumbo, zona o ruta del avión",0.9071,0.769,0.9189,OOF
6,futuro,lgb_future_clear_ses,+ segundo paso de sesión: cada tarea se trabaja una sola vez,0.9142,0.781,0.9248,OOF
7,futuro,suav_lgb_future_clear10_ses,A: 10 semillas + suavizado en la racha · público 0.947,0.9183,0.784,0.9292,OOF
8,futuro,f10_instr_l63_s10_ses,N: + bloque instr (tipo de instrucción que encaja con la tarea) + 63 hojas +...,0.9242,0.784,0.9292,OOF
9,futuro+oráculo,suav_lgb_future_clear10_orcoord_ses,B: A + oráculo de coordinaciones (otros controladores) · público 0.934,0.9259,0.796,0.9291,OOF


**Lectura:**
- **El pasado aporta ~1.5 puntos** (0.656 → 0.672). **El futuro aporta ~24** (0.672 → 0.90). Casi toda la nota sale de ver qué hace
  el controlador *después*: la siguiente instrucción que da al avión delata en qué estaba trabajando.
- Sobre el futuro, tres mejoras sólidas: `clear` (+0.6), el **segundo paso de sesión** (+0.7) y `instr` + árboles más grandes (+0.6).
- El **oráculo** (qué hicieron otros controladores del mismo escenario) suma +0.4–0.9 en CV, pero es la única fuente cuyo efecto
  varía mucho de un controlador a otro (sección 6).

## 3 · Por qué funciona lo que funciona

### 3a · La regla que aprende el modelo: "el avión que recibe una instrucción en los próximos 10 s"

Contribución media (SHAP del primer pase) de cada origen de información y efecto de los segundos hasta la siguiente
instrucción al avión (`futuro/pruebas/f9_explica.py`). El reparto es el mismo en CV, en el test del 020 y en el del 023.

In [7]:
f9 = ROOT / "futuro/pruebas/notas/f9_influencia.csv"
pd.read_csv(f9, index_col=[0, 1]).round(1) if f9.exists() else "(falta notas/f9_influencia.csv)"

futuro  pasado  snapshot  futuro clear  otras sesiones
modelo zona                                                            
M0     CV          33.4     5.8      60.8           0.0             0.0
       test 020    34.1     5.0      61.0           0.0             0.0
       test 023    34.2     4.5      61.4           0.0             0.0
A      CV          29.6     5.3      55.6           9.5             0.0
       test 020    30.1     5.2      54.7          10.0             0.0
       test 023    30.5     4.8      55.9           8.8             0.0
O      CV          26.8     4.1      44.2           8.7            16.2
       test 020    27.9     4.0      43.7           9.3            15.2
       test 023    28.5     3.6      44.6           7.7            15.7
B      CV          27.5     4.6      50.2           8.6             9.1
       test 020    28.0     4.3      48.4           9.5             9.8
       test 023    28.2     4.0      50.2           8.7             8.9

| Siguiente instrucción al avión | ≤ 3 s | 3–10 s | 10–30 s | 30–60 s | > 60 s |
|---|---|---|---|---|---|
| efecto medio sobre el score (modelo A) | +2.0 | +1.9 | +0.9 | ≈ 0 | −0.1 |

### 3b · El segundo paso de sesión

Un logit condicional de 9 pesos (`futuro/feat_session.py`) sobre el score del primer pase en las otras rachas de la sesión. El peso
más grande es negativo: **si la tarea ya fue la favorita en otra racha, aquí probablemente no lo es** ("explaining away", porque cada
tarea se trabaja una vez). Debe ser pequeño: un LightGBM en su lugar sobreajusta (0.906 frente a 0.914). A la red la empeora.

### 3c · `instr` y árboles de 63 hojas con `extra_trees`

`f_next_input` solo mira la primera instrucción siguiente; en el 16 % de las filas la instrucción que encaja con la tarea
(nivel → exit_conditions, rumbo → return_to_route o zone, transfer → transfer) no es la primera. Ninguno de los dos cambios gana por
separado; juntos, +0.6. Cada paso se ha medido con 3 y con 10 semillas: el ruido entre semillas es de ±0.2.

## 4 · Dónde falla y dónde está el techo

In [8]:
def por_tipo(nombre):
    o = oof(nombre)
    if o is None: return None
    h = hits(o)
    return h.groupby(resp.task_index.reindex(h.index).map(lambda i: TY[i])).mean()
P = pd.DataFrame({k: por_tipo(v) for k, v in [("limpia", "lgb_adri"), ("A", "suav_lgb_future_clear10_ses"),
                                              ("N", "f10_instr_l63_s10_ses"), ("NOR", "f10_instr_l63_or_s10_ses")]})
P["% de respuestas"] = resp.task_index.map(lambda i: TY[i]).value_counts(normalize=True) * 100
P.sort_values("% de respuestas", ascending=False).round(3)

,limpia,A,N,NOR,% de respuestas
task_index,,,,,
transfer_aircraft,0.863,0.986,0.986,0.987,26.770
assume_aircraft,0.838,0.984,0.984,0.984,25.548
zone_conflict,0.707,0.934,0.945,0.955,16.099
entry_coordination,0.384,0.748,0.796,0.785,8.106
return_to_route,0.506,0.895,0.905,0.909,7.112
exit_coordination,0.093,0.632,0.631,0.648,5.655
exit_conditions,0.179,0.830,0.835,0.853,3.400
conflict_resolution,0.300,0.895,0.892,0.892,2.960
qos_improvement,0.163,0.934,0.914,0.934,1.951


In [9]:
# los fallos son de tramo entero: si el modelo falla un tramo, lo falla en todos sus snapshots
o = oof("f10_instr_l63_or_s10_ses")
h = hits(o).rename("hit").to_frame().join(snap).join(resp.task_name)
tr = h.groupby(["block", "task_name"]).hit.agg(["mean", "size"])
fallos = (1 - h.hit).sum()
pd.Series({
    "acierto (NOR)": f"{h.hit.mean():.4f}",
    "tramos acertados enteros": f"{(tr['mean'] == 1).mean():.1%}",
    "tramos fallados enteros": f"{(tr['mean'] == 0).mean():.1%}",
    "% de los fallos que están en tramos fallados enteros": f"{(tr.loc[tr['mean'] == 0, 'size'].sum() / fallos):.1%}",
}).to_frame("NOR")

,NOR
acierto (NOR),0.9278
tramos acertados enteros,93.5%
tramos fallados enteros,4.7%
% de los fallos que están en tramos fallados enteros,79.9%


**Lo que dicen los diagnósticos** (`futuro/pruebas/f8_diag.py`, `f8_parejas.py`, `errores.py`, `auditoria_coord.py`):
- **La mitad de los fallos son coordinaciones** (entry/exit). Coordinar con un sector vecino no es una instrucción al avión: no deja
  huella en las columnas. El 77 % de los tramos fallados no tiene ninguna huella observable ni en el pasado ni en el futuro.
- **El 41 % de los fallos acierta el avión pero no la tarea.** En las tareas de acción la respuesta desaparece a los ~3 s; en las
  coordinaciones la señal **se invierte** (desaparece la hermana, el transfer o el assume, y la coordinación sigue 1–3 min).
  Un clasificador dedicado a desempatar tareas del mismo avión no supera al modelo (92.1 % frente a 94.3 %).
- **Ante la misma situación, dos controladores eligen la misma tarea solo el 12.7 % de las veces.** No hay colisiones (features
  idénticas con etiquetas distintas): el error es reducible en teoría, pero la información que falta no está en los datos.
- **La curva de aprendizaje no se aplana** (25/50/75/100 % de los controladores: 0.870 / 0.893 / 0.898 / 0.906): más
  controladores ayudarían; el modelo final, entrenado con los 18, debería rendir algo por encima de la CV.

## 5 · Por qué no funciona lo que no funciona

Todas estas pruebas están medidas con la misma CV; el detalle y las cifras exactas están en la sección indicada de
`futuro/LGB_futuro.ipynb` (LF) o en el script.

| Familia | Pruebas | Resultado | Por qué |
|---|---|---|---|
| Otros modelos | XGBoost, CatBoost, red (transformer) con futuro | 0.896 / 0.874 / 0.887 frente a 0.901 (LF 5, 6c) | con la información del futuro todos aprenden lo mismo y fallan en los mismos snapshots |
| Stacks y mezclas sin oráculo | media ponderada, meta-ranker, stacks con sesión | ±0.1 (LF 6c, 13c, 15) | los modelos están muy correlacionados (0.86 de acuerdo); los pesos van a uno solo |
| Parámetros | lento, regularizado, más o menos hojas, truncation, objetivos binary/xendcg | ruido (LF 2–3, 12a; `f1_lgb.py`) | manda la información, no el ajuste; binary/xendcg empeoran tras el segundo paso |
| Bloques de features | `coord`, `block`, `rel`, `geo`, `logic` y `past` con futuro | 0 a −0.4 (LF 6, 11a, 16, 18, 19a) | lo que dicen ya está en `f_task_end` o en los rangos; no hay posiciones de aviones |
| Post-procesos en la racha | HMM, Kalman, modelo por racha, transiciones aprendidas, asignación de episodios | = suavizado o peor (LF 15, 23b) | los tramos se aciertan o se fallan enteros: no hay ningún snapshot bueno del que tirar |
| Variantes del segundo paso | Sinkhorn, LightGBM, más variables, iteraciones | saturado (LF 9, 11b) | el logit de 9 pesos ya recoge la estructura |
| Adaptación al controlador | prior por EM, covariate shift, Bayes jerárquico, estilo del controlador, pseudo-etiquetas | ruido o peor (LF 11c, 15, 18b, 21a, 23) | el estilo del controlador no es estable fuera de muestra; ajustarlo sin etiquetas refuerza los errores |
| Especialistas | solo coordinaciones, solo tareas con instrucción | peso óptimo 0 (LF 16, 23a) | no saben cuándo la respuesta NO es de su tipo |
| Etiquetas | relevancia graduada, quitar respuestas aisladas, limpiar etiquetas | ruido; limpiar era fuga (`f6_soft.py`; LGB_adri 12b) | el ruido de etiqueta no se arregla entrenando |
| Línea limpia | XGBoost, objetivos, selección de variables, dos etapas, red | ±0.007 (`pruebas/h1–h5`, `NN_pablo`) | con un solo snapshot el techo está en ~0.66: el 83 % de los fallos son del avión equivocado |

## 6 · Público frente a privado y elección de las entregas

### 6a · El oráculo en el público

Kaggle solo enseña el público (controlador 020). Los envíos con oráculo bajaron en el público aunque la CV los daba por mejores.
Hipótesis H1 (formulada a posteriori con los 5 primeros envíos): cuantas más entry_coordination deja el modelo en el top-3 del 020,
mejor nota; el 020 coordina mucho y el oráculo saca entry_coordination del top-3. **Con el sexto envío deja de cumplirse:** NOR deja
menos entry_coordination que A y que `lgb_future` y aun así saca más (tabla de abajo). H1 es una pista débil, no una regla: el público
son 2 sesiones y ±1–2 puntos de ruido. El 023 casi no tiene entry_coordination como respuesta, así que este efecto pesa poco en el privado.

In [10]:
ENVIOS = {"lgb_future": ("futuro/submissions/sub_lgb_future.csv", 0.942),
          "A": ("futuro/submissions/sub_suav_lgb_future_clear10_ses.csv", 0.947),
          "oráculo completo": ("futuro/submissions/sub_suav_lgb_future_clear10_or_ses.csv", 0.940),
          "B": ("futuro/submissions/sub_suav_lgb_future_clear10_orcoord_ses.csv", 0.934),
          "N": ("futuro/pruebas/submissions/sub_f10_instr_l63_s10_ses.csv", 0.951),
          "NOR": ("futuro/pruebas/submissions/sub_f10_instr_l63_or_s10_ses.csv", 0.948)}
tipo = te.set_index(["sample_id", "row_index"]).task_index
at = te.drop_duplicates("sample_id").set_index("sample_id").atco
filas = []
for k, (f, pub) in ENVIOS.items():
    s = pd.read_csv(ROOT / f)
    top = s.assign(r=s.prediction.str.split()).explode("r").astype({"r": int})
    top["ec"] = [tipo.get((a, b)) == 2 for a, b in zip(top.sample_id, top.r)]
    g = top.groupby("sample_id").ec.any()
    filas.append(dict(envio=k, publico=pub, **{f"entry_coord en top-3 ({a:03d})": round(g[at.reindex(g.index) == a].mean() * 100, 1) for a in (20, 23)}))
pd.DataFrame(filas).sort_values("publico")

,envio,publico,entry_coord en top-3 (020),entry_coord en top-3 (023)
3,B,0.934,33.6,18.4
2,oráculo completo,0.940,36.7,22.7
0,lgb_future,0.942,44.0,25.6
1,A,0.947,47.7,26.9
5,NOR,0.948,41.6,22.6
4,N,0.951,54.0,26.4


### 6b · Qué pareja elegir

Al cierre se marcan 2 envíos y el privado (023) se puntúa con **el mejor de los dos**. Como no sabemos cómo es el 023, se simula con
los 18 controladores de la CV: para cada uno, el máximo de los dos aciertos; E[máx] es la media.

In [11]:
import itertools
M = {"A": "suav_lgb_future_clear10_ses", "B": "suav_lgb_future_clear10_orcoord_ses",
     "N": "f10_instr_l63_s10_ses", "NOR": "f10_instr_l63_or_s10_ses"}
H = pd.DataFrame({k: hits(oof(v)) for k, v in M.items()}).join(snap.atco)
g = H.groupby("atco").mean() * 100
pares = pd.DataFrame([dict(pareja=f"{x} + {y}", E_max=np.maximum(g[x], g[y]).mean(),
                           una_con_oraculo_y_otra_sin=(x in "AN") != (y in "AN"),
                           acuerdo=np.corrcoef(g[x], g[y])[0, 1]) for x, y in itertools.combinations(M, 2)])
pares.sort_values("E_max", ascending=False).round(3)

,pareja,E_max,una_con_oraculo_y_otra_sin,acuerdo
4,B + NOR,92.351,False,0.974
3,B + N,92.168,True,0.977
0,A + B,92.162,True,0.936
2,A + NOR,92.117,True,0.964
5,N + NOR,92.116,True,0.984
1,A + N,91.900,False,0.956


**Decisión: NOR + N.**
- La CV favorece llevar oráculo (mejor media, mejor peor controlador, y es información que nada más aporta).
- Pero los dos primeros envíos con oráculo bajaron en el público (NOR, sobre la base de N, ya no: 0.948), y la CV no puede ver un fallo
  sistemático en test.
- Con una entrega con oráculo (NOR) y otra sin él (N), Kaggle cubre los dos casos. Dos con oráculo (B + NOR) ganan solo ~0.2 en
  E[máx], dentro del ruido, y lo pierden todo si el oráculo falla en el 023.

**Lo que sabemos del privado:** el nivel, ~0.91–0.92 (unos 3 puntos por debajo del público: el 023 es un controlador más difícil,
con más snapshots sin instrucción próxima), con un intervalo de ±5 puntos que domina el propio controlador (calibración,
`futuro/pruebas/f12_calib.py`). Las diferencias entre modelos (±0.5) son mucho menores: no se puede saber de antemano cuál gana.

**Aviso de reglas:** la línea futuro usa otros snapshots de la sesión (regla 1 del equipo) y el oráculo usa etiquetas de otros
controladores y el escenario (regla 4). La entrega limpia de referencia es `submissions/sub_lgb_adri_v2.csv` (CV 0.664, público 0.642).

## 7 · Reproducir los envíos

```bash
# A y B (Pablo): LightGBM con futuro + clear, 10 semillas, + segundo paso de sesión (+ oráculo de coordinaciones para B)
python futuro/m_lgb_future.py --feats base,logic,past,future,clear --sesion --seeds 10 --name lgb_future_clear10
python futuro/m_lgb_future.py --feats base,logic,past,future,clear --sesion --oraculo --oraculo-var coord --seeds 10 --name lgb_future_clear10_orcoord
python futuro/pruebas/f5_stack.py suav lgb_future_clear10_ses lgb_future_clear10_orcoord_ses     # suavizado con CV anidada
# N y NOR (Juanma): + bloque instr, 63 hojas, extra_trees, 10 semillas, + sesión (+ oráculo para NOR)
bash futuro/pruebas/f10_final.sh                          # N
python futuro/pruebas/f10_instr.py cv f10_instr_l63_or 10 # NOR
# comparar dos OOF con el criterio robusto (media, mediana por sesión, controladores, bootstrap, escenarios)
python futuro/pruebas/comparar.py suav_lgb_future_clear10_ses f10_instr_l63_or_s10_ses
```
Las salidas de experimentos que no alimentan ningún envío se movieron fuera del repo (`~/FlowDataLab_archivo/`); sus cifras siguen en
`futuro/pruebas/resultados.jsonl` y en `experiments/*.csv`.